# 🚀 1-Click 8B Model Training for Agent 65 (Best & Fastest Method)

This notebook uses **Unsloth** (the fastest, most advanced fine-tuning engine in the world) to train your **8 Billion parameter model** on **Free Google Colab (T4 GPU)**.

- 🧠 **Model**: `Meta-Llama-3.1-8B-Instruct` (8 Billion parameters, Gemini/ChatGPT level reasoning)
- ⏱️ **Training Time**: ~15–20 minutes
- 💰 **Cost**: 100% Free
- 📦 **Output**: An updated model file (`agent65-8b-latest-q4_k_m.gguf`) ready to drop into Ollama on your PC with **zero API keys**!

---
### How to run:
1. Click **Runtime** -> **Change runtime type** -> Select **T4 GPU** -> Click **Save**.
2. Run each step from top to bottom (click the ▶️ play button on each cell).

### Step 1: Install High-Performance Training Stack
Click the play button ▶️ below. This installs the required AI libraries in ~2 minutes.

In [ ]:
!pip install --upgrade pip
!pip install unsloth unsloth_zoo
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl peft accelerate bitsandbytes datasets triton
print("\n✅ Training stack installed successfully!")

### Step 2: Load the 8B Parameter Foundation Model
Click ▶️ below. This downloads and sets up the 8 Billion parameter Llama 3.1 model in 4-bit precision.

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None # Auto detection
load_in_4bit = True

print("[*] Loading Meta Llama 3.1 8B Instruct... please wait ~2-3 minutes...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Attach LoRA parameter adapters (all projection layers)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("\n✅ 8B Model loaded and ready for training!")

### Step 3: Upload `training_data.jsonl`
Click ▶️ below. Click **Choose Files** and select `c:\StudentHelpdesk\BACKEND\training_data.jsonl` from your PC.

In [ ]:
from google.colab import files
import os, glob, json
from datasets import Dataset

print("Upload your `training_data.jsonl`:")
uploaded = files.upload()

jsonl_files = [f for f in glob.glob("*.jsonl")]
data_file = jsonl_files[0] if jsonl_files else "training_data.jsonl"
print(f"\n[*] Ingesting dataset: {data_file}")

# Load and normalize dialogues
dialogues = []
with open(data_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            entry = json.loads(line)
            msgs = entry.get("messages", [])
            if not msgs:
                continue
            
            # Ensure assistant response is present in messages
            if msgs[-1].get("role") != "assistant":
                resp = entry.get("model_response") or entry.get("gemini_response") or ""
                if resp:
                    msgs.append({"role": "assistant", "content": resp})
            
            if msgs[-1].get("role") == "assistant" and len(msgs) >= 2:
                dialogues.append({"messages": msgs})
        except Exception as e:
            continue

print(f"[*] Loaded {len(dialogues)} valid dialogue turns.")
raw_dataset = Dataset.from_list(dialogues)

def format_for_llama(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in convos]
    return {"text": texts}

train_dataset = raw_dataset.map(format_for_llama, batched=True)
print(f"✅ Successfully formatted {len(train_dataset)} conversations for the 8B model!")

### Step 4: Train the 8B Model
Click ▶️ below to start training. It will optimize all weights and adapt the 8B model to your student records.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Optimal for ~50-500 dialogue turns
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Training the 8B model... please wait...")
trainer_stats = trainer.train()
print("\n🎉 8B Model training complete!")

### Step 5: Export and Download for Ollama
Click ▶️ below. This exports your newly trained 8B model to a compact `.gguf` file and automatically downloads it to your computer!

In [ ]:
print("[*] Exporting 8B model to GGUF format for Ollama...")
model.save_pretrained_gguf("agent65_8b_finetuned", tokenizer, quantization_method = "q4_k_m")

import glob
from google.colab import files
gguf_files = glob.glob("agent65_8b_finetuned*/**/*.gguf", recursive=True) + glob.glob("*.gguf")
if gguf_files:
    target_gguf = gguf_files[0]
    print(f"\n[+] Triggering download for: {target_gguf} (~4.8 GB)...")
    files.download(target_gguf)
else:
    print("Look at the left Colab Files tab to download the generated .gguf file.")

### Step 6: Deploy on your Windows PC in 1 Command
Once downloaded to your PC:
1. Move the downloaded `.gguf` file into your folder:  
   `c:\StudentHelpdesk\BACKEND\models\agent65-8b-latest.gguf`
2. In your Windows PowerShell terminal, run:
   ```powershell
   ollama create agent65-8b:latest -f c:\StudentHelpdesk\BACKEND\models\Modelfile.agent65-8b
   ```
3. Your local AI is now completely updated with your newly trained 8B weights!